# Cosmic Engine — Shot-Graph / Stargate Sequence (manifest-driven)

This notebook exercises the architecture on branch `claude/keen-maxwell-sAB8n`:

- `scene/` — ShotGraph + Shot + Transition + Palette manifest
- `render/core.py` — Renderer ABC + name registry
- `render/adapters.py` — wraps gas-giant, Kerr black hole, slit-scan tunnel, exotic physics
- `render/kerr.py` — Kerr ray-tracer with Page–Thorne disk, frame-dragging, Doppler beaming
- `render/io.py` — ACES filmic + sRGB tonemap, EXR/MP4 writers
- `director/continuity.py` — histogram-matched crossfade + smoothstep blend
- `director/graph.py` — overlapping-transition graph executor
- `studio/` — artifact store, hand-coded metrics critic, mutation space, producer, session loop

**Runtime:** set the Colab runtime to **GPU** (T4 is fine). The fluid sim and Kerr renderer need it.

End-to-end pipeline this notebook drives:

1. Install deps and clone the branch.
2. Run the CPU test suite to verify the architecture imports cleanly.
3. Load `scene/examples/jupiter_to_stargate.json` (Jovian approach → Kerr anomaly → slit-scan tunnel).
4. Render it to MP4 via `GraphRunner` and display inline.
5. Tweak the manifest live (palette, Kerr spin, transition kind) and re-render.
6. Hand-author a fresh ShotGraph from scratch.
7. Use the studio artifact store + metrics critic to score a render and show keyframes.
8. Close the loop: explore the parameter space with a Producer + Session.

## 1. Install dependencies

In [ ]:
!pip -q install taichi 'imageio[ffmpeg,pyav]' av numpy
!apt-get -qq install -y ffmpeg > /dev/null
import taichi, imageio, av, numpy as np
print('taichi', taichi.__version__, '| imageio', imageio.__version__, '| av', av.__version__, '| numpy', np.__version__)

## 2. Clone the branch

Uses a shallow clone of the feature branch directly under `/content/cosmic_engine`. Re-running the cell pulls the latest commit on the same branch.

In [ ]:
import os, subprocess, sys

REPO_URL  = 'https://github.com/pmcray/cosmic_engine.git'
BRANCH    = 'claude/keen-maxwell-sAB8n'
REPO_DIR  = '/content/cosmic_engine'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

head = subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip()
print('HEAD:', head)

# Evict cached cosmic_engine modules so re-running cells after a git pull
# picks up the freshly cloned source instead of the previous import's bytecode.
# Jupyter does not invalidate sys.modules when files on disk change, which is
# what produces the misleading "old traceback through new source" symptom.
_PURGE_PREFIXES = ('render', 'director', 'scene', 'infinite_director', 'encounters', 'physics')
_purged = [m for m in list(sys.modules) if m.startswith(_PURGE_PREFIXES)]
for _m in _purged:
    del sys.modules[_m]
if _purged:
    print(f'cleared {len(_purged)} cached cosmic_engine module(s); next import will re-read disk')

## 3. Smoke test the architecture (CPU only)

All eight tests should pass. They cover manifest round-trip, camera interpolation, the palette registry, ACES tone-mapping behaviour, the histogram-matched crossfade, the transition engine, the example manifest loader, and an end-to-end `GraphRunner` run on the trivial constant renderer. **No GPU touched yet.**

In [ ]:
!python -m tests.test_manifest

## 4. Inspect the example manifest

In [ ]:
from scene.manifest import load_manifest
from scene.palette import PALETTES

MANIFEST_PATH = 'scene/examples/jupiter_to_stargate.json'
graph = load_manifest(MANIFEST_PATH)

print(f'Title: {graph.title}')
print('Shots:')
for s in graph.shots:
    print(f'  {s.id:18s} renderer={s.renderer:18s} dur={s.duration_frames:>3d}f @ {s.fps}fps  res={s.resolution} palette={s.palette.name}')
print('Transitions:')
for t in graph.transitions:
    print(f'  {t.from_shot:18s} -> {t.to_shot:18s} kind={t.kind:10s} dur={t.duration_frames}f params={t.params}')
print('\nAvailable palettes:', sorted(PALETTES.keys()))

## 5. Fast-mode override

Full 1920×1080 × 336 frames takes several minutes per shot on a T4 and the first Kerr render also pays a one-off Taichi JIT cost. The cell below downscales the in-memory graph for a sanity render that finishes in roughly a minute. Set `FAST_MODE = False` for the full version.

In [ ]:
FAST_MODE = True

if FAST_MODE:
    for s in graph.shots:
        s.resolution = (640, 360)
        s.duration_frames = max(24, s.duration_frames // 4)
    for t in graph.transitions:
        t.duration_frames = max(6, t.duration_frames // 3)

print('Effective render plan:')
for s in graph.shots:
    print(f'  {s.id:18s} {s.duration_frames:>3d}f  {s.resolution}')
for t in graph.transitions:
    print(f'  {t.from_shot} -> {t.to_shot:18s} {t.kind:10s} {t.duration_frames}f')

## 6. Render the example manifest end-to-end

First run also JIT-compiles every Taichi kernel. Expect a one-time 20–40 s warm-up before frames start rolling.

In [ ]:
import time, os
import render.adapters  # registers gas_giant, kerr_black_hole, slitscan_tunnel, exotic_physics
from director.graph import GraphRunner

os.makedirs('outputs', exist_ok=True)
out_path = 'outputs/jupiter_to_stargate.mp4'

t0 = time.time()
GraphRunner(graph, out_path).run()
dt = time.time() - t0
print(f'wrote {out_path} ({os.path.getsize(out_path) / 1024:.1f} KiB) in {dt:.1f}s')

## 7. Inline playback

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def show_mp4(path, width=720):
    data = open(path, 'rb').read()
    b64  = b64encode(data).decode()
    return HTML(
        f'<video width={width} controls autoplay loop muted playsinline>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'
    )

show_mp4(out_path)

## 8. Tweak the manifest live

The `graph` is just dataclasses — mutate it in memory and re-render. Here we swap to the JWST NIRCam palette and spin the Kerr black hole up to near-extremal `a/M = 0.99`.

In [ ]:
for s in graph.shots:
    s.palette.name = 'jwst_nircam'

kerr = graph.shot_by_id('kerr_anomaly')
kerr.params['spin']            = 0.99
kerr.params['inclination_deg'] = 87.0
kerr.params['disk_outer']      = 18.0

out_path2 = 'outputs/jupiter_to_stargate_jwst_spin99.mp4'
GraphRunner(graph, out_path2).run()
show_mp4(out_path2)

## 9. Hand-author a fresh ShotGraph

Build a two-shot sequence — near-extremal Kerr followed by a slit-scan tunnel — straight from Python, using the slit-scan family for the transition.

In [ ]:
from scene.manifest import Camera, PaletteRef, Shot, ShotGraph, Transition

my_graph = ShotGraph(
    title='kerr_into_stargate',
    shots=[
        Shot(
            id='kerr',
            renderer='kerr_black_hole',
            params={'spin': 0.95, 'inclination_deg': 85.0, 'disk_outer': 16.0, 'steps': 240},
            palette=PaletteRef(name='hubble_sii_ha_oiii'),
            duration_frames=48,
            fps=24,
            resolution=(640, 360),
            motion_hint='approach',
        ),
        Shot(
            id='tunnel',
            renderer='slitscan_tunnel',
            params={'time_scale': 0.06},
            palette=PaletteRef(name='trumbull_2001', intensity=1.2),
            duration_frames=64,
            fps=24,
            resolution=(640, 360),
            motion_hint='tunnel',
        ),
    ],
    transitions=[
        Transition(from_shot='kerr', to_shot='tunnel', kind='slitscan', duration_frames=18, params={'intensity': 1.4}),
    ],
)
my_graph.validate()

out_path3 = 'outputs/custom_kerr_into_stargate.mp4'
GraphRunner(my_graph, out_path3).run()
show_mp4(out_path3)

## 10. Studio: artifact store + metrics critic

Every `GraphRunner.run()` can now be steered into a self-contained, addressable artifact directory: the manifest, provenance (git SHA, library versions), the video, per-shot keyframes (`.npy` for exact metrics + tonemapped `.png` for humans and VLMs), and a metrics sidecar. A deterministic `MetricsCritic` then scores any artifact for temporal coherence, palette adherence, dynamic range, edge density, and common failure modes (NaN runs, frozen frames, near-degenerate flats).

The keyframe strip rendered below is what a future Claude-vision critic will receive as its visual input.

In [ ]:
from studio import ArtifactStore, MetricsCritic
from pathlib import Path
from IPython.display import HTML, display
from base64 import b64encode

store = ArtifactStore("renders")
art_path = GraphRunner(graph, "_unused.mp4", artifact_store=store, artifact_label="baseline").run()
print("artifact:", art_path)

result = MetricsCritic().evaluate_artifact(art_path)
agg = result.aggregate
print(f"\naggregate composite score : {agg['composite_score']:.3f}")
print(f"worst shot                : {agg['worst_shot']['id']}  ({agg['worst_shot']['composite_score']:.3f})")
print()
print(f"  {'shot':<18s} {'score':>6s} {'palette_d':>10s} {'temporal':>10s} {'edge':>8s} {'notes'}")
for sid, m in result.shots.items():
    notes = "; ".join(m.notes) if m.notes else "-"
    print(f"  {sid:<18s} {m.composite_score:>6.3f} {m.palette_distance:>10.4f} {m.temporal_abs_diff:>10.5f} {m.edge_density:>8.4f} {notes}")


def show_keyframes(artifact_dir, shot_id, height=110):
    kdir = Path(artifact_dir) / "shots" / shot_id / "keyframes"
    pngs = sorted(kdir.glob("*.png"))
    if not pngs:
        return HTML(f"<p>no keyframes for {shot_id}</p>")
    imgs = []
    for p in pngs:
        b64 = b64encode(p.read_bytes()).decode()
        imgs.append(f'<img src="data:image/png;base64,{b64}" style="height:{height}px; margin:2px;" />')
    return HTML(
        f'<div style="margin:6px 0;"><strong>{shot_id}</strong>'
        f'<div style="display:flex;flex-wrap:wrap;">{"".join(imgs)}</div></div>'
    )

for sid in result.shots:
    display(show_keyframes(art_path, sid))

## 11. Producer + Session loop (closing the critic loop)

The critic earns its keep only when something *acts* on its scores. A `MutationSpace` declares which knobs the loop may turn; a `Producer` picks one knob per iteration and perturbs it within bounds; a `Session` renders each candidate at proxy resolution, scores it via the critic, caches duplicates, and tracks the best so far.

The cell below explores the hand-authored two-shot graph from section 9 (`my_graph`). Expect the first proposal to pay the Kerr JIT cost; subsequent ones run much faster. The proxy keeps each render under 30 s on a T4.

In [ ]:
from studio import (
    ArtifactStore, MetricsCritic, MutationSpace, RandomProducer,
    Session, shot_param, shot_palette, transition_kind,
)

space = MutationSpace([
    shot_param("kerr",   "spin",            kind="float",  bounds=(0.2, 0.999), perturb_scale=0.25),
    shot_param("kerr",   "inclination_deg", kind="float",  bounds=(45.0, 89.0), perturb_scale=0.2),
    shot_param("kerr",   "disk_outer",      kind="float",  bounds=(8.0, 22.0),  perturb_scale=0.2),
    shot_palette("kerr",   choices=("trumbull_2001", "hubble_sii_ha_oiii", "jwst_nircam")),
    shot_palette("tunnel", choices=("trumbull_2001", "jwst_nircam")),
    transition_kind("kerr", "tunnel", choices=("crossfade", "slitscan", "match_cut")),
])

session = Session(
    base_graph=my_graph,
    store=ArtifactStore("renders/session"),
    producer=RandomProducer(my_graph, space, seed=42),
    critic=MetricsCritic(),
    proxy={"resolution": (320, 180), "duration_scale": 0.5},
)

def _on(att):
    tag = "  (cached)" if att.cached else ""
    desc = f"{att.mutation['param']}: {att.mutation['from']} -> {att.mutation['to']}"
    print(f"  iter {att.iteration:>2}  score {att.score:.3f}  {desc}{tag}")

print("Exploring at 320x180 proxy:")
session.explore(budget=6, label_prefix="lab1", on_attempt=_on)
print()

best = session.best()
print(f"Winner: iter {best.iteration}  score {best.score:.3f}")
print(f"Artifact: {best.artifact_path}")
log_path = session.save("renders/session_log.json")
print(f"Session log written to {log_path}")

from IPython.display import display
for shot in my_graph.shots:
    display(show_keyframes(best.artifact_path, shot.id))

## 12. Where to go next

- **Render the winning manifest at full quality.** `session.render_best_at_full_quality(artifact_label="master")` reconstructs the winning shot params at the base graph's resolution and frame count.
- **Try the epsilon-greedy producer.** Swap `RandomProducer` for `EpsilonGreedyProducer(my_graph, space, epsilon=0.3, seed=42)` to bias exploration toward perturbations of the current best.
- **Bump up quality on section 6.** Set `FAST_MODE = False` and re-run cells 6, 12, 16 for the full 1920×1080 master.
- **Tune the critic.** `studio/metrics.py` exposes the weights in `_COMPOSITE_WEIGHTS` and the bounds in `_normalize` — they are intentionally simple so they can be replaced with values learned from human ratings.
- **Add a VLM critic.** A thin wrapper around Claude vision (Anthropic API) reading the keyframes + manifest and returning structured per-shot scores plus one actionable suggestion. Slot it in via `Session(critic=...)`.
- **Add a renderer.** Implement a subclass of `render.core.Renderer`, decorate it with `@register_renderer('your_name')`, and reference it from a Shot. Saturn-class gas giant, volumetric nebula, neutron star — anything from the roadmap fits this slot.
- **Inspect any artifact from the shell.** `python -m studio.cli list`, `python -m studio.cli evaluate <path>`, `python -m studio.cli show <path>`.